In [1]:
import os
import json

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

from openai import OpenAI

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

with open("../keys.json", "r") as fi:
    api_key = json.load(fi)["api_key"]

## Retrieval-Augmented Generation

In this exercise, you'll put together a RAG system and compare outputs from RAG vs. just querying an LLM.

For this exercise, you'll be asking about Subspace-Constrained LoRA (SC-LoRA), a new technique described in [a recent article publised on arXiv.org](https://arxiv.org/abs/2505.23724). You've been provided the text of this article in the file 2505.23724v1.txt.

First, you'll need to set up a way to interact with the generator model. You can use the OpenAI class from the openai library for this. See [this page](https://developers.openai.com/api/docs/guides/text?lang=python) for more information. When you do this, you'll need to set the base_url to "https://openrouter.ai/api/v1" and to pass in your api key. Set the model to "openrouter/owl-alpha".

First, ask the model "How does SC-LoRA differ from regular LoRA?" without providing any additional context. Read through a few different responses. **Question:** Are the responses accurate, or does it seem like the model is just making up something that sounds plausible?

In [2]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

In [4]:
query = "How does SC-LoRA differ from regular LoRA?"

response = client.responses.create(
    model="openrouter/owl-alpha",
    input=[
        {"role": "user", "content": query}
    ]
)

print(response.output_text)

SC-LoRA (Scaling and Coherence-aware Low-Rank Adaptation) differs from regular LoRA (Low-Rank Adaptation) in several key ways, primarily focusing on how it handles the adaptation process and the resulting model performance. Here's a breakdown of the main differences:

### 1. **Rank Selection and Scaling**
*   **Regular LoRA:** Typically uses a fixed, often small, rank for the low-rank matrices (A and B) that are added to the pre-trained weights. The choice of rank is a hyperparameter that balances efficiency and performance.
*   **SC-LoRA:** Introduces a more sophisticated approach to rank selection and scaling. It often involves:
    *   **Dynamic Rank Allocation:** Instead of a uniform rank across all layers or modules, SC-LoRA might dynamically allocate different ranks to different parts of the model based on their importance or sensitivity to the adaptation task. This allows for more efficient use of parameters.
    *   **Scaling Factors:** It might incorporate specific scaling fac

### Part 1: Manual RAG

In order to get more reliable responses, let's set up a RAG system.

In this first part, you'll build all of the pieces of the RAG system individually.

First, you'll need the retriever portion. Create a FAISS index to hold the text of the article. Encode this text using the all-MiniLM-L6-v2 encoder. Note that you'll want to divide the text into smaller chunks rather than encoding the whole article all at once. You could try, for example, the [RecursiveCharacterTextSplitter class from LangChain](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter). You'll need to specify a chunk_size and chunk_overlap. You could try a chunk_size of 500 and overlap of 50 as a starting point.

In [5]:
with open("../data/2505.23724v1.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [8]:
print(text[10000:10500])

et al., 2021). For large language models, pre-
serving world knowledge remains challenging due
to massive pre-training data and model size. Recent
efforts mitigate forgetting by freezing pre-trained
layers while introducing new adapters (Wu et al.,
2024; Dou et al., 2024). Recently (Yang et al.,
2024) proposed CorDA with Knowledge-Preserved
Adaptation (KPA) mode, addressing world knowl-
edge forgetting through LoRA initialization.
3 Method
Below, we first review the vanilla LoRA, and de-
scribe 


In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [15]:
chunks = text_splitter.split_text(text)

In [16]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
embeddings = embedder.encode(chunks, normalize_embeddings=True)

In [19]:
embeddings

array([[-0.02755547, -0.00032351,  0.01251429, ..., -0.08669789,
        -0.00168366,  0.09532458],
       [ 0.046388  , -0.10269721,  0.03018901, ..., -0.01732611,
        -0.00658801,  0.06322885],
       [-0.02480439, -0.04846335, -0.03130866, ..., -0.0103746 ,
         0.02160847,  0.07295156],
       ...,
       [-0.05202908,  0.01141351,  0.06638309, ...,  0.0751631 ,
        -0.06254414,  0.02023483],
       [-0.0847427 , -0.10415659, -0.03010891, ...,  0.03028686,
        -0.09369642, -0.01963139],
       [-0.11524055, -0.01378046, -0.05905307, ..., -0.0733518 ,
        -0.06647401,  0.03610926]], shape=(112, 384), dtype=float32)

In [18]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

Next, use the following as a system prompt:

```
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise. "
    f"Context: {context}"
)
```

In [40]:
query = "How does ABC-LoRA differ from regular LoRA?"

query_embedding = embedder.encode([query])

_, I = index.search(query_embedding, k = 5)

In [41]:
I[0]

array([55, 44, 50, 24, 39])

In [42]:
context = "\n\n".join([chunks[i] for i in I[0]])

In [43]:
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise. "
    f"Context: {context}"
)

In [44]:
print(system_prompt)

Use the given context to answer the question. If you don't know the answer, say you don't know. Use three sentences maximum and keep the answer concise. Context: LoRA 320M 46.81 1.05 7.04 18.30 41.77 5.46 23.62
PiSSA 320M 47.44 3.32 6.84 19.20 51.63 7.70 29.67
CorDA IPA 320M 30.20 9.83 5.41 15.15 51.40 8.34 29.87
CorDA KPA 320M 46.21 10.64 7.33 21.39 45.03 6.54 25.79
SC-LoRAβ= 0 320M 44.26 5.18 7.19 18.88 53.53 8.98 31.25
β= 0.5 320M 48.91 7.70 6.89 21.17 53.37 8.62 31.00
β= 0.8 320M 50.52 10.64 7.04 22.73 52.46 7.62 30.04
Table 3: Results of world knowledge preservation and math ability after fine-tuning on MetaMATH.

sponses (score = 5) as harmfulness rate . Lower
values for both metrics indicate stronger safety of
the model.
5Method #Params HS↓HR(%) ↓Utility ↑
Llama-2-7b-Chat - 1.100 1.212 24.13
Full fine-tuning 6738M 1.364 5.455 51.41
LoRA 320M 1.176 2.424 50.32
PiSSA 320M 1.252 4.242 51.87
CorDA IPA 320M 1.209 3.333 44.61
CorDA KPA 320M 1.106 0.606 50.89
SC-LoRAβ= 0.5 320M 1.161 1

Use the FAISS index to pull in relevant context to fill in the context. Try passing in this additional system prompt. Hint: you can do this by using the following messages in the client.chat.completions.create function

```
    messages=[
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": query,
        }
    ]
```

How does adding this context change the results?

In [45]:
messages=[
    {
        "role": "system",
        "content": system_prompt,
    },
    {
        "role": "user",
        "content": query,
    }
]

In [46]:
messages

[{'role': 'system',
  'content': "Use the given context to answer the question. If you don't know the answer, say you don't know. Use three sentences maximum and keep the answer concise. Context: LoRA 320M 46.81 1.05 7.04 18.30 41.77 5.46 23.62\nPiSSA 320M 47.44 3.32 6.84 19.20 51.63 7.70 29.67\nCorDA IPA 320M 30.20 9.83 5.41 15.15 51.40 8.34 29.87\nCorDA KPA 320M 46.21 10.64 7.33 21.39 45.03 6.54 25.79\nSC-LoRAβ= 0 320M 44.26 5.18 7.19 18.88 53.53 8.98 31.25\nβ= 0.5 320M 48.91 7.70 6.89 21.17 53.37 8.62 31.00\nβ= 0.8 320M 50.52 10.64 7.04 22.73 52.46 7.62 30.04\nTable 3: Results of world knowledge preservation and math ability after fine-tuning on MetaMATH.\n\nsponses (score = 5) as harmfulness rate . Lower\nvalues for both metrics indicate stronger safety of\nthe model.\n5Method #Params HS↓HR(%) ↓Utility ↑\nLlama-2-7b-Chat - 1.100 1.212 24.13\nFull fine-tuning 6738M 1.364 5.455 51.41\nLoRA 320M 1.176 2.424 50.32\nPiSSA 320M 1.252 4.242 51.87\nCorDA IPA 320M 1.209 3.333 44.61\nCorDA K

In [47]:
response = client.responses.create(
    model="openrouter/owl-alpha",
    input=messages
)

print(response.output_text)

Based on the provided context, there is no mention of a method called "ABC-LoRA." The text discusses several variants of LoRA, such as **PiSSA**, **CorDA** (IPA and KPA), and **SC-LoRA**, but does not define or describe ABC-LoRA. Therefore, I cannot answer how ABC-LoRA differs from regular LoRA.


### Part 2: LangChain

You can also use the [LangChain library](https://www.langchain.com/) to help build your RAG system.

For the retriever, you can use the [HugginFaceEmbeddings class](https://reference.langchain.com/python/langchain-huggingface/embeddings/huggingface/HuggingFaceEmbeddings), using the all-MiniLM-L6-v2 model, to create your embedding model. There is also a [FAISS class](https://reference.langchain.com/python/langchain-community/vectorstores/faiss/FAISS), which has a useful from_texts method. Once you've created your vector store, use the [as_retriever method](https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html#langchain_community.vectorstores.faiss.FAISS.as_retriever) on it and save it to a variable named `retriever`.

In [48]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [49]:
vector_store = FAISS.from_texts(chunks, embedding_model)

In [50]:
retriever = vector_store.as_retriever()

In [51]:
retriever.invoke(query)

[Document(id='9a336e54-bec3-4f01-867f-c7b78aead056', metadata={}, page_content='LoRA 320M 46.81 1.05 7.04 18.30 41.77 5.46 23.62\nPiSSA 320M 47.44 3.32 6.84 19.20 51.63 7.70 29.67\nCorDA IPA 320M 30.20 9.83 5.41 15.15 51.40 8.34 29.87\nCorDA KPA 320M 46.21 10.64 7.33 21.39 45.03 6.54 25.79\nSC-LoRAβ= 0 320M 44.26 5.18 7.19 18.88 53.53 8.98 31.25\nβ= 0.5 320M 48.91 7.70 6.89 21.17 53.37 8.62 31.00\nβ= 0.8 320M 50.52 10.64 7.04 22.73 52.46 7.62 30.04\nTable 3: Results of world knowledge preservation and math ability after fine-tuning on MetaMATH.'),
 Document(id='3010a1e5-379f-4285-a4b5-a5fb91e5f085', metadata={}, page_content='sponses (score = 5) as harmfulness rate . Lower\nvalues for both metrics indicate stronger safety of\nthe model.\n5Method #Params HS↓HR(%) ↓Utility ↑\nLlama-2-7b-Chat - 1.100 1.212 24.13\nFull fine-tuning 6738M 1.364 5.455 51.41\nLoRA 320M 1.176 2.424 50.32\nPiSSA 320M 1.252 4.242 51.87\nCorDA IPA 320M 1.209 3.333 44.61\nCorDA KPA 320M 1.106 0.606 50.89\nSC-LoRAβ=

For the generator, you can use the [ChatOpenAI class](https://python.langchain.com/docs/integrations/chat/openai/). Be sure to set base_url="https://openrouter.ai/api/v1", model_name="openrouter/owl-alpha", and openai_api_key= Your API key. Save this to a variable named `llm`.

In [52]:
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    model_name="openrouter/owl-alpha",
    openai_api_key=api_key
)

Now that the two components have been created, we can combine them into a chat template using the [ChatPromptTemplate class](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html). We can set up a system prompt and the pass that in, like
```
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)
```

In [57]:
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

In [58]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

rag_chain = (
    {
        "context": (
            RunnableLambda(lambda x: x["question"])
            | retriever
            | RunnableLambda(format_docs)
        ),
        "question": RunnableLambda(
            lambda x: x["question"]
        )
    }
    | prompt
    | llm
)

In [60]:
query = "How does SC-LoRA differ from regular LoRA?"

response = rag_chain.invoke(
    {"question": query}
)

print(response.content)

SC-LoRA is a LoRA initialization method that focuses on preserving knowledge during fine-tuning, whereas regular LoRA does not prioritize this. SC-LoRA achieves better utility and safety alignment compared to regular LoRA, which shows a sharp decline in safety as learning rate increases. However, SC-LoRA does not strongly constrain updates during fine-tuning, which can lead to a drop in knowledge preservation over more complex tasks.


Now we need to connect all of the pieces together. Newer versions of LangChain commonly use LCEL (LangChain Expression Language)[https://www.langchain.com/blog/langchain-expression-language] to build pipelines where components are connected together using the | operator.

You'll need:

* A helper function that combines retrieved documents into a single string of context
* A pipeline that:
    - extracts the question from the input
    - sends the question to the retriever
    - formats the retrieved documents into context
    - passes both the context and question into the prompt
    - sends the completed prompt to the LLM

A simplified diagram looks like this:

input
   ↓
extract question
   ↓
retrieve documents
   ↓
format context
   ↓
prompt
   ↓
LLM

You can create this chain by using 

```
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

rag_chain = (
    {
        "context": (
            RunnableLambda(lambda x: x["question"])
            | retriever
            | RunnableLambda(format_docs)
        ),
        "question": RunnableLambda(
            lambda x: x["question"]
        )
    }
    | prompt
    | llm
)
```

Take a minute to study this and see if you can figure out how the syntax works.

Finally, invoke your chain using:

```
response = rag_chain.invoke(
    {"question": query}
)

print(response.content)
```

Compare the output from this section with both previous approaches:
* LLM without retrieval
* Manual RAG
* LangChain RAG

The quality of the answers from (2) and (3) should be similar. The purpose of LangChain is not to improve the model itself. Rather, it provides abstractions that simplify the retrieval and generation workflow.